In [7]:
import torch
import os

In [11]:
from booknlp.booknlp import BookNLP

In [12]:
import os

model_params = {
    "pipeline": "entity,quote",
    "model": "custom",
    "entity_model_path": "/home/ccarrol2/booknlp_models/entities_google_bert_uncased_L-6_H-768_A-12-v1.0.model",
    "coref_model_path": "/home/ccarrol2/booknlp_models/coref_google_bert_uncased_L-12_H-768_A-12-v1.0.model",
    "quote_attribution_model_path": "/home/ccarrol2/booknlp_models/speaker_google_bert_uncased_L-12_H-768_A-12-v1.0.1.model",
    "bert_model_path": "/home/ccarrol2/.cache/huggingface/hub/",
}

# sanity check
for k in ["entity_model_path", "coref_model_path", "quote_attribution_model_path"]:
    print(k, os.path.isfile(model_params[k]), model_params[k])

entity_model_path True /home/ccarrol2/booknlp_models/entities_google_bert_uncased_L-6_H-768_A-12-v1.0.model
coref_model_path True /home/ccarrol2/booknlp_models/coref_google_bert_uncased_L-12_H-768_A-12-v1.0.model
quote_attribution_model_path True /home/ccarrol2/booknlp_models/speaker_google_bert_uncased_L-12_H-768_A-12-v1.0.1.model


In [ ]:
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def strip_position_ids(model_file, out_file=None):
    if out_file is None:
        if model_file.endswith(".model"):
            out_file = model_file[:-6] + "_nopos.model"
        else:
            out_file = model_file + "_nopos"

    sd = torch.load(model_file, map_location=device)

    if "bert.embeddings.position_ids" in sd:
        print("Removing bert.embeddings.position_ids from:", model_file)
        del sd["bert.embeddings.position_ids"]
    else:
        print("No position_ids key in:", model_file)

    torch.save(sd, out_file)
    print("Saved patched model to:", out_file)
    return out_file

# Patch every model path present in model_params
for key in ["entity_model_path", "coref_model_path", "quote_attribution_model_path"]:
    path = model_params.get(key)
    if isinstance(path, str) and os.path.isfile(path):
        model_params[key] = strip_position_ids(path)
    else:
        print("Skipping", key, "->", path)

print("\nUpdated model_params:")
print(model_params)

In [15]:
booknlp=BookNLP("en", model_params)


{'pipeline': 'entity,quote', 'model': 'custom', 'entity_model_path': '/home/ccarrol2/booknlp_models/entities_google_bert_uncased_L-6_H-768_A-12-v1.0_nopos.model', 'coref_model_path': '/home/ccarrol2/booknlp_models/coref_google_bert_uncased_L-12_H-768_A-12-v1.0_nopos.model', 'quote_attribution_model_path': '/home/ccarrol2/booknlp_models/speaker_google_bert_uncased_L-12_H-768_A-12-v1.0.1_nopos.model', 'bert_model_path': '/home/ccarrol2/.cache/huggingface/hub/'}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 61712.86it/s]


--- startup: 5.954 seconds ---


In [16]:
input_folder = "/home/ccarrol2/dialogue/benchmark_github_repo/nightmare/benchmark/manual_samples"

output_directory= "/home/ccarrol2/dialogue/benchmark_github_repo/nightmare/pipeline_tests/nightmare/booknlp_nightmare_output"


for filename in os.listdir(input_folder):
    input_file = os.path.join(input_folder, filename)
    book_id = filename
    booknlp.process(input_file, output_directory, book_id)






--- spacy: 0.279 seconds ---
--- entities: 0.706 seconds ---
--- quotes: 0.002 seconds ---
--- attribution: 0.565 seconds ---
--- name coref: 0.002 seconds ---
--- TOTAL (excl. startup): 2.329 seconds ---, 4274 words
--- spacy: 0.248 seconds ---
--- entities: 0.558 seconds ---
--- quotes: 0.002 seconds ---
--- attribution: 0.669 seconds ---
--- name coref: 0.001 seconds ---
--- TOTAL (excl. startup): 2.242 seconds ---, 3637 words
--- spacy: 0.230 seconds ---
--- entities: 0.546 seconds ---
--- quotes: 0.002 seconds ---
--- attribution: 0.348 seconds ---
--- name coref: 0.002 seconds ---
--- TOTAL (excl. startup): 1.907 seconds ---, 3399 words
--- spacy: 0.252 seconds ---
--- entities: 0.590 seconds ---
--- quotes: 0.002 seconds ---
--- attribution: 0.386 seconds ---
--- name coref: 0.001 seconds ---
--- TOTAL (excl. startup): 2.216 seconds ---, 3527 words
--- spacy: 0.258 seconds ---
--- entities: 0.543 seconds ---
--- quotes: 0.002 seconds ---
--- attribution: 0.203 seconds ---
--- na

## Apply BookNLP to Nightmare

In [17]:
def normalise(text):
    """Reduce a string to its visible, non-quotation characters.

    Drops all whitespace (spaces, tabs, newlines) and every quotation mark
    listed in QUOTE_CHARS, so similarity reflects spoken content rather than
    punctuation choices or layout differences between pipelines.

    Examples
    --------
    >>> normalise('"Oh dear!" she said.')
    'Ohdear!shesaid.'
    >>> normalise("  'Oh dear!'  ")
    'Ohdear!'
    """
    QUOTE_CHARS = set("\"'`\u201c\u201d\u2018\u2019\u00ab\u00bb\u2039\u203a\u201e\u201f")
    return "".join(ch for ch in text if not ch.isspace() and ch not in QUOTE_CHARS)


def load_gold_strings(ann_path):
    strings = {}

    books_list = os.listdir(
        "/home/ccarrol2/dialogue/benchmark_github_repo/nightmare/pipeline_tests/nightmare/booknlp_nightmare_output/quotes"
    )
    books_list = [x.split(".")[0] for x in books_list]

    with open(ann_path, "r", encoding="utf-8") as f:
        lines = [line.rstrip("\n").split("\t") for line in f]

    for book in books_list:
        texts = []
        target = book + ".txt"   # match the text file column format

        for parts in lines:
            if len(parts) < 4:
                continue
            if parts[2] != "speech":
                continue
            if parts[3] != target:
                continue

            norm = normalise(parts[1])   # tagged text column
            if norm:
                texts.append(norm)

        strings[book] = texts

    return strings

In [18]:
import shutil

In [19]:
quotes_files = []

input_folder = "/home/ccarrol2/dialogue/benchmark_github_repo/nightmare/pipeline_tests/nightmare/booknlp_nightmare_output"

for filename in os.listdir(input_folder):
    if filename.endswith(".quotes"):

        rootname = filename[:-11]
        print(rootname)

        target_dir = os.path.join(input_folder, rootname)
        os.makedirs(target_dir, exist_ok=True)

        src = os.path.join(input_folder, filename)
        dst = os.path.join(target_dir, filename)
        shutil.move(src, dst)

        #x = output_path + rootname
        #os.mkdir(x)
        #os.replace(filename, x)


514_manual_sample
840_manual_sample
84_manual_sample
2046_manual_sample
38830_manual_sample
41646_manual_sample
883_manual_sample
11_manual_sample
54683_manual_sample
6124_manual_sample
40814_manual_sample
17862_manual_sample


In [20]:
import csv

In [21]:
def load_predictions(data_path):
    """Load BookNLP predictions from a directory of per-book folders.

    Parameters
    ----------
    data_path : str
        Root directory containing one subfolder per book, each with a
        <folder_name>.quotes TSV file inside it.
    ann_dir : str
        Path to gold .ann files (unused for loading, but available if needed
        for cross-referencing).

    Returns
    -------
    dict[str, list[str]]
        {book_name: [raw_quote_string, ...]}
        book_name has "_sampled" stripped so it matches the gold .ann naming.
    """
    predictions = {}

    for folder in sorted(os.listdir(data_path)):
        book_dir = os.path.join(data_path, folder)
        if not os.path.isdir(book_dir):
            continue

        quotes_path = os.path.join(book_dir, folder + ".txt.quotes")
        if not os.path.exists(quotes_path):
            print(f"[booknlp] Skipping {folder}: no .quotes file")
            continue

        # Strip "_sampled" so keys align with gold file names.
        # e.g. "jane_eyre_sampled" -> "jane_eyre"
        book_name = folder.replace("_sampled", "")

        strings = []

        with open(quotes_path, "r", encoding="utf-8") as f:
            # BookNLP .quotes files are tab-delimited with a header row.
            # quoting=QUOTE_NONE because the quote text itself contains
            # quotation-mark characters that would confuse the CSV parser.
            reader = csv.DictReader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
            for row in reader:
                text = row.get("quote", "") or ""
                if text.strip():
                    strings.append(text)

        predictions[book_name] = strings

    return predictions

In [22]:
preds = load_predictions("/home/ccarrol2/dialogue/benchmark_github_repo/nightmare/pipeline_tests/nightmare/booknlp_nightmare_output/quotes")

In [23]:
print(preds)

{'11_manual_sample': ['“ Curiouser and curiouser ! ”', '“ now I ’m opening out like the largest telescope that ever was ! Good - bye , feet ! ”', '“ Oh , my poor little feet , I wonder who will put on your shoes and stockings for you now , dears ? I ’m sure I sha n’t be able ! I shall be a great deal too far off to trouble myself about you : you must manage the best way you can;—but I must be kind to them , ”', '“ or perhaps they wo n’t walk the way I want to go ! Let me see : I ’ll give them a new pair of boots every Christmas . ”', '“ They must go by the carrier , ”', '“ and how funny it ’ll seem , sending presents to one ’s own feet ! And how odd the directions will look !', '”', '“ You ought to be ashamed of yourself , ”', '“ a great girl like you , ”', '“ to go on crying in this way ! Stop this moment , I tell you ! ”', '“ Oh ! the Duchess , the Duchess ! Oh ! wo n’t she be savage if I ’ve kept her waiting ! ”', '“ If you please , sir — ”', '“ Dear , dear ! How queer everything is

In [24]:
# If your dict is called pred
pred_norm = {
    book: [normalise(s) for s in strings]
    for book, strings in preds.items()
}

In [25]:
print(pred_norm)

{'11_manual_sample': ['Curiouserandcuriouser!', 'nowImopeningoutlikethelargesttelescopethateverwas!Good-bye,feet!', 'Oh,mypoorlittlefeet,Iwonderwhowillputonyourshoesandstockingsforyounow,dears?ImsureIshantbeable!Ishallbeagreatdealtoofarofftotroublemyselfaboutyou:youmustmanagethebestwayyoucan;—butImustbekindtothem,', 'orperhapstheywontwalkthewayIwanttogo!Letmesee:IllgivethemanewpairofbootseveryChristmas.', 'Theymustgobythecarrier,', 'andhowfunnyitllseem,sendingpresentstoonesownfeet!Andhowoddthedirectionswilllook!', '', 'Yououghttobeashamedofyourself,', 'agreatgirllikeyou,', 'togooncryinginthisway!Stopthismoment,Itellyou!', 'Oh!theDuchess,theDuchess!Oh!wontshebesavageifIvekeptherwaiting!', 'Ifyouplease,sir—', 'Dear,dear!Howqueereverythingisto-day!Andyesterdaythingswentonjustasusual.IwonderifIvebeenchangedinthenight?Letmethink:wasIthesamewhenIgotupthismorning?IalmostthinkIcanrememberfeelingalittledifferent.ButifImnotthesame,thenextquestionis,WhointheworldamI?Ah,thatsthegreatpuzzle!', 'Ims

In [26]:
ann_path = "/home/ccarrol2/dialogue/benchmark_github_repo/nightmare/benchmark/final_benchmark.tsv"

nightmare = load_gold_strings(ann_path)

In [27]:
print(nightmare)

{'883_manual_sample': ['Amerenothing.Ihadaheadache—gonenow—andwasnotquitefitforahottheatre,soIstayedathome.Iaskedyouifyouwerenotwell,becauseyoulooksowhite.', 'Againstyou?', 'Anhonestdeclaration!', 'AndIbegleavetoaskyou,MrRokesmith,', 'Atleast,sir,', 'BecauseIamtruly,deeply,profoundlyinterestedinyou,MissWilfer.BecauseIwishtoseeyoualwaysatyourbest.BecauseI—shallIgoon?', 'Believeme,', 'Dearme!Areyounotwell,MrRokesmith?', 'DoIbelieveso?', 'DoI?Ihavehadabusyevening.', 'HaveIdoneso?', 'Iammuchrelievedtohearit,', 'Idoubtifyoueverwill.Good-night.OfcourseIshallbecarefultoconcealanytracesofthisinterviewfromMrandMrsBoffin,aslongasIremainhere.Trustme,whatyouhavecomplainedofisatanendforever.', 'Ihavefarotherviewsinlife,andwhyshouldyouwasteyourown?', 'Ihopeyouarewronginthatimpression.Ishouldbeverysorrytohavejustifiedit.IthinkIhavenot.Forthefuturethereisnoapprehension.Itisallover.', 'Imustrequest,sir,', 'Inowfeelobligedtogoon,', 'Irejectit,sir,', 'IshouldbeblindanddeafifIwerenotpreparedforthereply.Fo

In [28]:
from pprint import pprint

pprint(nightmare, width=120, sort_dicts=True)

{'11_manual_sample': ['Ahem!',
                      'Areyou—areyoufond—of—ofdogs?',
                      'AsifIwouldtalkonsuchasubject!Ourfamilyalwayshatedcats:nasty,low,vulgarthings!Dontletmehearthenameagain!',
                      'Aswetasever,',
                      'Butshemusthaveaprizeherself,youknow,',
                      'Butwhohaswon?',
                      'Butwhoistogivetheprizes?',
                      'Comeupagain,dear!',
                      'Curiouserandcuriouser!',
                      'Dear,dear!Howqueereverythingisto-day!Andyesterdaythingswentonjustasusual.IwonderifIvebeenchangedinthenight?Letmethink:wasIthesamewhenIgotupthismorning?IalmostthinkIcanrememberfeelingalittledifferent.ButifImnotthesame,thenextquestionis,WhointheworldamI?Ah,thatsthegreatpuzzle!',
                      'Didyouspeak?',
                      'EdwinandMorcar,theearlsofMerciaandNorthumbria,declaredforhim:andevenStigand,thepatrioticarchbishopofCanterbury,founditadvisable—',
             

In [29]:
from pprint import pprint

pprint(pred_norm, width=120, sort_dicts=True)

{'11_manual_sample': ['Curiouserandcuriouser!',
                      'nowImopeningoutlikethelargesttelescopethateverwas!Good-bye,feet!',
                      'Oh,mypoorlittlefeet,Iwonderwhowillputonyourshoesandstockingsforyounow,dears?ImsureIshantbeable!Ishallbeagreatdealtoofarofftotroublemyselfaboutyou:youmustmanagethebestwayyoucan;—butImustbekindtothem,',
                      'orperhapstheywontwalkthewayIwanttogo!Letmesee:IllgivethemanewpairofbootseveryChristmas.',
                      'Theymustgobythecarrier,',
                      'andhowfunnyitllseem,sendingpresentstoonesownfeet!Andhowoddthedirectionswilllook!',
                      '',
                      'Yououghttobeashamedofyourself,',
                      'agreatgirllikeyou,',
                      'togooncryinginthisway!Stopthismoment,Itellyou!',
                      'Oh!theDuchess,theDuchess!Oh!wontshebesavageifIvekeptherwaiting!',
                      'Ifyouplease,sir—',
                      'Dear,dear!Howqueer

In [30]:
def best_similarity(needle, haystack):
    """Highest RapidFuzz ratio between *needle* and any string in *haystack*.

    Single-pair convenience helper (used by evaluate_book's edge cases where
    predicted or gold is empty). For the main scoring path, evaluate_book()
    below uses rapidfuzz.process.cdist to score every pair at once instead of
    calling this in a Python loop.
    """
    if not needle or not haystack:
        return 0.0
    result = extractOne(needle, haystack, scorer=fuzz.ratio)
    return result[1] / 100.0 if result else 0.0

In [31]:
def evaluate_book(predicted, gold, threshold):
    """Score one novel. Both `predicted` and `gold` should already be normalised.

    Returns (counts_dict, predicted_detail_list).

    counts_dict has keys: TP, FP, FN, n_pred, n_gold, matched_gold.
    predicted_detail is a list of dicts with {string, best_similarity, correct}
    so that per-book JSON output shows exactly why each prediction passed/failed.

    Scoring logic
    -------------
    - Precision direction: for each predicted string, find its best match in gold.
      If similarity >= threshold, it's a TP; otherwise FP.
    - Recall direction: for each gold string, find its best match in predicted.
      If nothing reaches the threshold, it's an FN (a missed gold quote).

    The two directions are independent — multiple predictions can match the same
    gold string, which is standard for fuzzy set-overlap scoring.

    Uses rapidfuzz.process.cdist to compute the full (n_pred x n_gold) similarity
    matrix in one compiled C++ call — no Python loops over string pairs. Precision
    and recall are both read off the matrix with numpy max operations.
    """
    if not predicted and not gold:
        return {"TP": 0, "FP": 0, "FN": 0, "n_pred": 0, "n_gold": 0,
                "matched_gold": 0}, []

    if not predicted:
        return {"TP": 0, "FP": 0, "FN": len(gold), "n_pred": 0,
                "n_gold": len(gold), "matched_gold": 0}, []

    if not gold:
        pred_detail = [{"string": p, "best_similarity": 0.0, "correct": False}
                       for p in predicted]
        return {"TP": 0, "FP": len(predicted), "FN": 0, "n_pred": len(predicted),
                "n_gold": 0, "matched_gold": 0}, pred_detail

    thresh_pct = threshold * 100  # rapidfuzz scores are 0-100

    # One compiled call computes every (predicted[i], gold[j]) pair.
    # Shape: (n_pred, n_gold), dtype float32, values 0-100.
    scores = cdist(predicted, gold, scorer=fuzz.ratio, dtype=np.float32)

    # ── Precision ─────────────────────────────────────────────────────────────
    best_pred_scores = scores.max(axis=1)          # shape (n_pred,)
    correct_mask     = best_pred_scores >= thresh_pct
    tp = int(correct_mask.sum())
    fp = len(predicted) - tp

    # ── Recall ────────────────────────────────────────────────────────────────
    best_gold_scores = scores.max(axis=0)          # shape (n_gold,)
    matched_gold     = int((best_gold_scores >= thresh_pct).sum())
    fn = len(gold) - matched_gold

    pred_detail = [
        {
            "string":          p,
            "best_similarity": round(float(best_pred_scores[i]) / 100, 4),
            "correct":         bool(correct_mask[i]),
        }
        for i, p in enumerate(predicted)
    ]

    counts = {
        "TP": tp, "FP": fp, "FN": fn,
        "n_pred": len(predicted), "n_gold": len(gold),
        "matched_gold": matched_gold,
    }
    return counts, pred_detail


In [33]:
!pip install rapidfuzz


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.14/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [32]:
import os
import json
import numpy as np
from rapidfuzz import fuzz
from rapidfuzz.process import cdist, extractOne

ModuleNotFoundError: No module named 'rapidfuzz'

In [ ]:
evaluate_book(pred_norm, nightmare, 0.9)

In [ ]:
def prf1(tp, fp, fn):
    """Compute precision, recall, and F1 from raw counts.

    Returns (0, 0, 0) when denominators are zero, which happens when a book
    has no predictions or no gold annotations.
    """
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f

In [ ]:
def write_report(book_results, tot, tot_gold, empty_gold, threshold, report_path, pipeline_name):
    """Format and write the human-readable evaluation report."""
    book_results.sort(key=lambda b: b["name"])
    col = max((len(b["name"]) for b in book_results), default=40)

    header = (
        f"{'Book':<{col}}  {'Pred':>5} {'Gold':>5} "
        f"{'TP':>4} {'FP':>4} {'FN':>4}  "
        f"{'Prec':>6} {'Rec':>6} {'F1':>6}"
    )
    lines = [
        f"=== {pipeline_name}: character-level dialogue match (threshold={threshold}) ===",
        "",
        "Dialogue is the positive class. A predicted quote is correct (TP) when its best",
        "character-level similarity to any gold quote in the same novel is >= threshold.",
        "FN = gold quotes with no prediction match at threshold.",
        "",
        "=== Per-Novel Results ===",
        header,
        "-" * len(header),
    ]

    for b in book_results:
        lines.append(
            f"{b['name']:<{col}}  {b['n_pred']:>5} {b['n_gold']:>5} "
            f"{b['TP']:>4} {b['FP']:>4} {b['FN']:>4}  "
            f"{b['precision']:>6.3f} {b['recall']:>6.3f} {b['f1']:>6.3f}"
        )

    p, r, f = prf1(tot["TP"], tot["FP"], tot["FN"])
    pg, rg, fg = prf1(tot_gold["TP"], tot_gold["FP"], tot_gold["FN"])

    lines += [
        "",
        "=== Corpus Totals (micro-averaged over all quotes) ===",
        f"Predicted quotes: {tot['n_pred']}   Gold quotes: {tot['n_gold']}",
        f"TP={tot['TP']}  FP={tot['FP']}  FN={tot['FN']}",
        f"Precision: {p:.3f}   Recall: {r:.3f}   F1: {f:.3f}",
        f"Unmatched gold quotes (missed dialogue): {tot['FN']}",
        "",
        f"=== Excluding {len(empty_gold)} books with empty gold lists ===",
        "(These books have no gold quote strings in the nightmare dictionary,",
        "so predictions there can only add false positives.)",
        f"Books excluded: {', '.join(empty_gold) if empty_gold else 'none'}",
        f"TP={tot_gold['TP']}  FP={tot_gold['FP']}  FN={tot_gold['FN']}",
        f"Precision: {pg:.3f}   Recall: {rg:.3f}   F1: {fg:.3f}",
    ]

    report = "\n".join(lines)
    print("\n" + report)
    with open(report_path, "w", encoding="utf-8") as fh:
        fh.write(report + "\n")
    print(f"\nReport written to {report_path}")

In [ ]:
def run_evaluation_dicts(predictions, gold_dict, results_dir, threshold=0.90, pipeline_name="pipeline"):
    """
    predictions: dict[str, list[str]]  -> e.g., pred_norm
    gold_dict:   dict[str, list[str]]  -> e.g., nightmare
    """
    import os, json

    per_book_dir = os.path.join(results_dir, "per_book")
    os.makedirs(per_book_dir, exist_ok=True)

    book_results = []
    tot = {"TP": 0, "FP": 0, "FN": 0, "n_pred": 0, "n_gold": 0}
    tot_gold = {"TP": 0, "FP": 0, "FN": 0, "n_pred": 0, "n_gold": 0}
    empty_gold = []

    # evaluate over shared keys (or all prediction keys if you prefer)
    common_books = sorted(set(predictions.keys()) & set(gold_dict.keys()))

    for book_name in common_books:
        predicted = predictions.get(book_name, [])
        gold = gold_dict.get(book_name, [])

        counts, pred_detail = evaluate_book(predicted, gold, threshold)

        for k in tot:
            tot[k] += counts[k]

        if counts["n_gold"] == 0:
            empty_gold.append(book_name)
        else:
            for k in tot_gold:
                tot_gold[k] += counts[k]

        p, r, f = prf1(counts["TP"], counts["FP"], counts["FN"])
        book_results.append({
            "name": book_name, **counts,
            "precision": p, "recall": r, "f1": f,
        })

        with open(os.path.join(per_book_dir, book_name + ".json"), "w", encoding="utf-8") as fh:
            json.dump(
                {"threshold": threshold, **counts, "precision": p, "recall": r, "f1": f, "predictions": pred_detail},
                fh, indent=2, ensure_ascii=False
            )

    report_path = os.path.join(results_dir, "evaluation_results.txt")
    write_report(book_results, tot, tot_gold, empty_gold, threshold, report_path, pipeline_name)

In [ ]:
results_dir = os.getcwd()

run_evaluation_dicts(
    predictions=pred_norm,
    gold_dict=nightmare,
    results_dir=results_dir,
    threshold=0.90,
    pipeline_name="booknlp"
)